# 🎮 Why Does That NPC Remember You Stole From Them?
### *A 7-minute crash course on Classes & Inheritance — through Game AI*

**AP CSA connection:** Unit 5 (Writing Classes) + Unit 9 (Inheritance)

---

Think of a game like *Skyrim* or *Zelda*. If you steal from a shopkeeper, walk away, come back an hour later — **they still remember**. If you fight a guard and run away hurt, they don't magically heal back to full health next time you see them.

That "remembering" is not magic. It's **one object, holding onto its own data, over time.**

That's it. That's the whole secret. Let's build it — in Java, exactly like you will on the AP exam.


## 1️⃣ An NPC's Memory = Instance Variables  *(⏱ ~1 min)*

**Instance variables** are private fields that belong to *one object*, not the whole program.

For an NPC, that "memory" is things like: how much health it has, where it's standing, and what it's currently doing.

Run the cell below 👇 — notice each NPC keeps its **own** health total.


In [ ]:
public class NPC {
    private String name;
    private int health;
    private int x, y;
    private String state;

    public NPC(String name, int startX, int startY) {
        // constructor: every NPC is born with a valid, complete state
        this.name = name;
        this.health = 100;
        this.x = startX;
        this.y = startY;
        this.state = "WANDER";
    }

    public void describe() {
        System.out.println(name + ": " + health + " HP, at (" + x + "," + y + "), state=" + state);
    }
}

NPC guard = new NPC("Guard Bob", 10, 4);
NPC villager = new NPC("Villager Sam", 2, 2);

guard.describe();
villager.describe();


**Notice:** `guard` and `villager` are two separate objects — each with its *own* copy of `health`, `x`, `y`, and `state`. That separation is exactly what makes state persistence possible: each NPC is its own little save file.


## 2️⃣ Encapsulation: Why "Private" Actually Matters *(⏱ ~1.5 min)*

Here's where real games get bugs. If `health` were `public`, *anything* could reach in and set it directly — no rules enforced.

```java
guard.health = -50;   // COMPILE ERROR if health is private — that's the point!
```

That line won't even compile, because `health` is `private`. That's not Java being annoying — that's Java **protecting your save file** from an impossible state (negative HP). The only legal way to change health is through a method *you* wrote, that *you* control.


In [ ]:
public class SafeNPC {
    private String name;
    private int health;

    public SafeNPC(String name) {
        this.name = name;
        this.health = 100;
    }

    public void takeDamage(int amount) {
        health -= amount;
        if (health < 0) {
            health = 0;   // clamp it -- no negative HP allowed, ever
        }
        System.out.println(name + " took " + amount + " damage -> " + health + " HP left");
    }

    public int getHealth() {
        return health;
    }
}

SafeNPC boss = new SafeNPC("Dragon King");
boss.takeDamage(30);
boss.takeDamage(9999);   // even a huge hit can't break the rules


**This is the single clearest real-world reason AP CSA insists on `private` fields + public methods instead of public fields.** Sloppy encapsulation in a classroom exercise costs you rubric points. Sloppy encapsulation in a shipped game causes real, visible bugs — a health bar showing "-50," a companion whose loyalty score exceeds the max, a save file with impossible values.


## 3️⃣ Constructors: Every NPC Needs a Valid Starting State *(⏱ ~30 sec)*

Look back at the `NPC(String name, int startX, int startY)` method above — that's the **constructor**. Its whole job is to guarantee no NPC is ever "born broken": no missing health, no undefined position, no null state. Every field gets a sane default the moment `new NPC(...)` runs.


## 4️⃣ Not All NPCs Think Alike: Inheritance *(⏱ ~1.5 min)*

A `Guard` and a `Civilian` are *both* NPCs — same basic memory (health, position) — but they react to seeing the player **completely differently**.

Instead of copy-pasting the whole class twice, we write ONE **abstract superclass**, and let each **subclass** override just the one method that differs: `decideAction()`.


In [ ]:
public abstract class BaseNPC {
    protected String name;
    protected int health = 100;

    public BaseNPC(String name) {
        this.name = name;
    }

    // abstract: every subclass MUST provide its own version
    public abstract String decideAction(boolean playerVisible);
}

public class Guard extends BaseNPC {
    public Guard(String name) { super(name); }

    @Override
    public String decideAction(boolean playerVisible) {
        if (playerVisible) {
            return name + ": ATTACK!";
        }
        return name + ": patrolling...";
    }
}

public class Civilian extends BaseNPC {
    public Civilian(String name) { super(name); }

    @Override
    public String decideAction(boolean playerVisible) {
        if (playerVisible) {
            return name + ": fleeing in terror!";
        }
        return name + ": wandering the market.";
    }
}


In [ ]:
import java.util.ArrayList;

ArrayList<BaseNPC> npcs = new ArrayList<BaseNPC>();
npcs.add(new Guard("Bob"));
npcs.add(new Civilian("Sam"));
npcs.add(new Guard("Rex"));

for (BaseNPC npc : npcs) {
    System.out.println(npc.decideAction(true));
}


Look closely at that `for` loop: it just calls `npc.decideAction(true)` — it never checks "is this a `Guard` or a `Civilian`?" Java automatically dispatches to the **correct overridden version** for each object at runtime.

**That's polymorphism** — the loop stays simple forever, even if you add a `Boss`, a `Merchant`, or 50 more NPC subclasses next month. This is the exact "why do we even bother with polymorphism" moment AP CSA is trying to teach.


## 5️⃣ The Twist: Saving This to Disk *(⏱ ~30 sec)*

When you save an `ArrayList<BaseNPC>` full of mixed subtypes to a file and reload it, the game has to remember not just *that* something is a `BaseNPC`, but **which subclass** it was — otherwise `decideAction()` can't be restored correctly. That's usually solved with a hidden "type tag" field. Inheritance isn't just an abstract OOP nicety — it directly shapes how save/load systems have to work.


## 🧠 Your Turn — Quick Challenge *(⏱ ~1 min)*

Write a **`Boss`** class that `extends BaseNPC`. When the player is visible, `decideAction` should return a taunt of your choice. Fill in the `...` below and run it.


In [ ]:
public class Boss extends BaseNPC {
    public Boss(String name) { super(name); }

    @Override
    public String decideAction(boolean playerVisible) {
        if (playerVisible) {
            return name + ": ...your taunt here...";
        }
        return name + ": sleeping on a pile of gold.";
    }
}

Boss finalDragon = new Boss("Final Dragon");
System.out.println(finalDragon.decideAction(true));


## ✅ Recap *(⏱ ~15 sec)*

| Game AI concept | AP CSA Unit | Idea |
|---|---|---|
| NPC's `health`, `x`, `y`, `state` fields | **Unit 5** | Instance variables = an object's private, persistent memory |
| `takeDamage()` clamping health at 0 | **Unit 5** | Encapsulation prevents corrupted state |
| `NPC(...)` constructor | **Unit 5** | Guarantees a valid starting state |
| `Guard` / `Civilian` / `Boss` `extends BaseNPC` | **Unit 9** | Inheritance + method overriding |
| One loop calls `decideAction()` on any NPC type | **Unit 9** | Polymorphism |

**Big idea:** A "smart" NPC isn't smart because of some fancy algorithm — it's smart because it's a *well-designed object* that remembers who it is.
